In [ ]:
import torch
import cv2
import numpy as np
import math
import matplotlib.pyplot as plt
import glob
import os
import pandas as pd
import segmentation_models_pytorch as smp 

# ================= CONFIGURATION =================
# 1. Path settings
MODEL_PATH = "titanium_unet_weights.pth" 
IMAGE_FOLDER = ##  location of images to process with model 
OUTPUT_CSV_NAME = "titanium_analysis_results.csv"

# 2. Scale Settings
MICRONS_PER_PIXEL = 0.02336448  # <--- UPDATE THIS VALUE
SCALE_BAR_LENGTH_MICRONS = 10 

# 3. Processing Settings
PATCH_SIZE = 256
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ================= HELPER FUNCTIONS =================

def opencv_skeletonize(img):
    """Robust skeletonization using only OpenCV"""
    img = img.copy()
    skel = np.zeros(img.shape, np.uint8)
    element = cv2.getStructuringElement(cv2.MORPH_CROSS, (3,3))
    while True:
        eroded = cv2.erode(img, element)
        temp = cv2.dilate(eroded, element)
        temp = cv2.subtract(img, temp)
        skel = cv2.bitwise_or(skel, temp)
        img = eroded.copy()
        if cv2.countNonZero(img) == 0:
            break
    return skel

def measure_intercept_average(mask, num_lines=100):
    """
    Calculates the Mean Linear Intercept (MLI) by casting random lines
    across the mask and counting phase transitions.
    """
    h, w = mask.shape
    total_length_pixels = 0
    total_intercepts = 0
    
    for _ in range(num_lines):
        x1, y1 = np.random.randint(0, w), np.random.randint(0, h)
        x2, y2 = np.random.randint(0, w), np.random.randint(0, h)
        
        length_px = int(np.hypot(x2 - x1, y2 - y1))
        if length_px < 10: continue 
        
        x_values = np.linspace(x1, x2, length_px).astype(int)
        y_values = np.linspace(y1, y2, length_px).astype(int)
        
        x_values = np.clip(x_values, 0, w-1)
        y_values = np.clip(y_values, 0, h-1)
        
        profile = mask[y_values, x_values]
        transitions = np.sum(np.abs(np.diff(profile)) > 0)
        
        total_intercepts += transitions
        total_length_pixels += length_px

    
    if total_intercepts == 0:
        return 0, total_length_pixels, total_intercepts
    
    mli = total_length_pixels / total_intercepts
    return mli, total_length_pixels, total_intercepts

def add_scale_bar(image, um_per_px, bar_length_um):
    """Draws a scale bar and text on the image."""
    h, w = image.shape[:2]
    bar_pixels = int(bar_length_um / um_per_px)
    margin = 40
    start_point = (w - margin - bar_pixels, h - margin)
    end_point = (w - margin, h - margin)
    
    color = 255 if image.ndim == 2 else (255, 255, 255)
    thickness = 6
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        color = (0, 255, 0)
        
    cv2.line(image, start_point, end_point, color, thickness)
    text = f"{bar_length_um} um"
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1.0
    text_size = cv2.getTextSize(text, font, font_scale, 2)[0]
    text_x = end_point[0] - (bar_pixels // 2) - (text_size[0] // 2)
    text_y = start_point[1] - 15
    cv2.putText(image, text, (text_x, text_y), font, font_scale, color, 2)
    return image

def process_large_image(image_path, model, patch_size=256, stride=128):
    original_img = cv2.imread(image_path)
    if original_img is None: return None, None
    img_rgb = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    pad_h = (math.ceil(h / patch_size) * patch_size) - h
    pad_w = (math.ceil(w / patch_size) * patch_size) - w
    img_padded = cv2.copyMakeBorder(img_rgb, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)
    ph, pw = img_padded.shape[:2]
    
    full_mask_acc = np.zeros((ph, pw), dtype=np.float32)
    count_map = np.zeros((ph, pw), dtype=np.float32)
    
    with torch.no_grad():
        for y in range(0, ph - patch_size + 1, stride):
            for x in range(0, pw - patch_size + 1, stride):
                patch = img_padded[y:y+patch_size, x:x+patch_size]
                
                input_tensor = patch.transpose(2, 0, 1).astype('float32') / 255.0
                input_tensor = torch.from_numpy(input_tensor).unsqueeze(0).to(DEVICE)
                
                logits = model(input_tensor)
                prob_mask = torch.sigmoid(logits).squeeze().cpu().numpy()
                
                full_mask_acc[y:y+patch_size, x:x+patch_size] += prob_mask
                count_map[y:y+patch_size, x:x+patch_size] += 1.0

    final_mask_prob = full_mask_acc / np.maximum(count_map, 1)
    final_mask_prob = final_mask_prob[:h, :w]
    final_mask_binary = (final_mask_prob > 0.5).astype(np.uint8) * 255
    
    return original_img, final_mask_binary

# ================= MAIN EXECUTION =================

def main():
    print(f"Loading model on {DEVICE}...")
    
    model = smp.Unet(
        encoder_name="resnet18",
        encoder_weights=None, 
        in_channels=3,
        classes=1,
        activation=None 
    )
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
    model.to(DEVICE)
    model.eval() 
    
    extensions = ["*.tif", "*.jpg", "*.png", "*.bmp"]
    image_files = []
    for ext in extensions:
        image_files.extend(glob.glob(os.path.join(IMAGE_FOLDER, ext)))
    image_files = [f for f in image_files if "_processed" not in f]
    
    print(f"Found {len(image_files)} images to process.")
    results_data = []

    for idx, img_path in enumerate(image_files):
        filename = os.path.basename(img_path)
        print(f"[{idx+1}/{len(image_files)}] Processing {filename}...")
        
        original, mask = process_large_image(img_path, model)
        if mask is None: continue 
            
        # --- MEASUREMENTS ---
        
        white_pixels = cv2.countNonZero(mask)
        total_pixels = mask.shape[0] * mask.shape[1]
        alpha_fraction = (white_pixels / total_pixels) * 100
        
        dist_map = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
        skeleton = opencv_skeletonize(mask)
        widths_px = dist_map[skeleton > 0] * 2
        widths_px = widths_px[widths_px > 2.0]
        widths_microns = widths_px * MICRONS_PER_PIXEL
        
        if len(widths_microns) > 0:
            avg_width = np.mean(widths_microns)
            std_width = np.std(widths_microns)
        else:
            avg_width = 0
            std_width = 0

       
        avg_intercept_px, total_length_px, total_intercepts = measure_intercept_average(mask, num_lines=30)
        
        # Convert pixel metrics to microns
        avg_intercept_um = avg_intercept_px * MICRONS_PER_PIXEL
        total_length_um = total_length_px * MICRONS_PER_PIXEL

        # --- SAVING OUTPUTS ---
        base_name = os.path.splitext(filename)[0]
        
        mask_visual = mask.copy()
        mask_with_scale = add_scale_bar(mask_visual, MICRONS_PER_PIXEL, SCALE_BAR_LENGTH_MICRONS)
        save_mask_path = os.path.join(IMAGE_FOLDER, f"{base_name}_processed.png")
        cv2.imwrite(save_mask_path, mask_with_scale)
        
        if len(widths_microns) > 0:
            plt.figure(figsize=(6, 4))
            plt.hist(widths_microns, bins=50, color='teal', edgecolor='black', alpha=0.7)
            
            plt.title(rf"Lath Widths: {filename}\nAvg: {avg_width:.2f} $\mu$m | Intercept: {avg_intercept_um:.2f} $\mu$m")
            plt.xlabel(r"Width ($\mu$m)")
            
            plt.ylabel("Count")
            plt.grid(axis='y', alpha=0.5)
            graph_path = os.path.join(IMAGE_FOLDER, f"{base_name}_processed_lathmeasurements.png")
            plt.savefig(graph_path)
            plt.close()
        
        
        results_data.append({
            "Filename": filename,
            "Scale (um/px)": MICRONS_PER_PIXEL,
            "Alpha Phase (%)": round(alpha_fraction, 2),
            "Avg Lath Width (um)": round(avg_width, 4),
            "Std Dev Width (um)": round(std_width, 4),
            "Intercept Average (um)": round(avg_intercept_um, 4),
            "Total Line Length (um)": round(total_length_um, 2),
            "Total Intercepts": int(total_intercepts)
        })

    if results_data:
        df = pd.DataFrame(results_data)
        output_path = os.path.join(IMAGE_FOLDER, OUTPUT_CSV_NAME)
        df.to_csv(output_path, index=False)
        print(f"\nBatch Complete. Data saved to: {output_path}")
        print(df.head())
    else:
        print("No data collected.")

if __name__ == "__main__":
    main()